[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/lab08_midterm_app.ipynb) · [Course home](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/00_course_home.ipynb)

**Google Colab version.** **Before you start: File → Save a copy in Drive.** Work only in your copy (its title starts with *Copy of*); changes to this original are not saved. Rename your copy `Lab 8 — <your last name>` (click the title at the top left).

**To submit:** **Runtime → Run all** and check that every cell ran, then **File → Save**. Click
**Share** (top right), add **steven.elliott@student.fairfield.edu** as **Editor**, and send. Colab keeps your saved versions
under **File → Revision history**; **File → Save and pin revision** keeps one permanently, which is how you
mark a submission.

**This week you submit two links:** your notebook (shared as **Editor**, as above) and the **view-only link** to the app (section 2 shows how).

**Coming back later?** Colab forgets variables and downloaded files when the runtime disconnects (after about 90 minutes idle). Open your copy and click **Runtime → Run all**.

# Lab 8 — Midterm: communicating for decisions
*Week 8. **Midterm.** Guide and worked example: sections 1–4. Your app: section 5. Rubric and checklist:
sections 6–7. Powerup: section 8.*

Your midterm is a **Colab notebook built for a reader**: an app that answers your capstone question for someone
who will never see your code: a hospital board member, a state health department director, a reporter. It is
version one of your capstone. A decision-maker needs the answer, the context that makes it mean something, and
an honest word about what the number cannot tell them. Not a data dump.

This notebook is a guide and a template. Sections 1–4 walk through a worked example built on the CMS data you
already know. Section 5 is where you build your own. The rubric is in section 6, so you know exactly what is
being measured.

**How labs work.** Save a copy in Drive, work in your copy, share it back with the instructor
when you submit. Every lab has a **baseline** (everyone finishes it in the session) and
**powerups** (optional depth). AI assistants may write code for you; **you own every line** and
may be asked to explain any of it. Cells marked `# TODO` are yours to fill in. This week you submit the
**view-only app link** as well as the notebook.

In [ ]:
# --- Course setup: run this cell first -------------------------------------------
# Loaders for the public datasets this course uses. The canonical copy of this code
# is labs/course_data.py in the course repo; you will read it line by line in week 3.
import io, json, urllib.request, urllib.parse
from pathlib import Path
import pandas as pd
import duckdb

CACHE = Path("data"); CACHE.mkdir(exist_ok=True)

def sql(query: str) -> pd.DataFrame:
    """Run SQL over any DataFrame in this notebook by its variable name (DuckDB)."""
    return duckdb.sql(query).df()

def _get(url, params=None, timeout=120) -> bytes:
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "healthcare-analytics-course/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- CMS Care Compare (data.cms.gov/provider-data) --------------------------------
# File names change at every CMS refresh; dataset identifiers do not. We ask the
# catalog for the current CSV URL every time instead of hard-coding a file name.
CMS_CATALOG = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
CMS_IDS = {
    "hospital_general_info": "xubh-q36u",   # one row per hospital
    "hrrp":                  "9n3s-kdb3",   # Hospital Readmissions Reduction Program, one row per hospital x measure
    "unplanned_visits":      "632h-zaca",   # readmissions / ED visits, one row per hospital x measure
    "complications_deaths":  "ynj2-r877",
    "timely_effective":      "yv7e-xc69",
}

def cms_csv_url(identifier: str) -> str:
    # Ask for the one dataset by id; the full keyword search can take minutes or time out.
    item = json.loads(_get(f"{CMS_CATALOG}/{identifier}", timeout=30))
    for d in item.get("distribution", []):
        if d.get("mediaType") == "text/csv":
            return d["downloadURL"]
    raise KeyError(f"No CSV distribution found for CMS dataset {identifier}")

# Course mirror: a snapshot of every CMS file the labs use, for when CMS is down or rate-limits
# a classroom downloading the same file at once. The live source is always tried first.
COURSE_DATA_BASE = "https://raw.githubusercontent.com/stevenelliottjr/healthcare-analytics-course-data/main/data"

def _fetch_or_mirror(path: Path, fetch, mirror_file: str) -> None:
    """Save `path` from the live source; if that fails, from the course mirror."""
    try:
        path.write_bytes(fetch())
    except Exception as e:
        print(f"Live download failed ({type(e).__name__}); using the course mirror for {path.name}.")
        path.write_bytes(_get(f"{COURSE_DATA_BASE}/{mirror_file}", timeout=300))

def load_cms(name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download (once) and read a Care Compare file. Everything is read as TEXT on purpose:
    IDs keep their leading zeros and 'Not Available' stays visible instead of silently
    becoming NaN. Converting columns to numbers is your job, and part of the lesson."""
    path = CACHE / f"cms_{name}.csv"
    if not path.exists():
        _fetch_or_mirror(path, lambda: _get(cms_csv_url(CMS_IDS[name]), timeout=300), f"cms/cms_{name}.csv")
    kw = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")
    kw.update(read_csv_kwargs)
    return pd.read_csv(path, **kw)

def to_num(s: pd.Series) -> pd.Series:
    """Text -> number; anything that is not a number ('Not Available', 'N/A', '') becomes NaN."""
    return pd.to_numeric(s, errors="coerce")

# ---- CDC PLACES, county release (Socrata API on data.cdc.gov) ---------------------
PLACES_COUNTY = "https://data.cdc.gov/resource/swc5-untb.json"
PLACES_SELECT = ("year,stateabbr,statedesc,locationname,locationid,categoryid,measureid,measure,"
                 "data_value,low_confidence_limit,high_confidence_limit,totalpopulation")

def load_places(measure_ids, data_value_type="Age-adjusted prevalence", state=None, page=50000) -> pd.DataFrame:
    """County-level model-based prevalence estimates (percent of adults). locationid is the
    5-digit county FIPS. Note: Connecticut reports *planning regions*, not counties, since 2022."""
    where = "measureid in(" + ",".join(f"'{m}'" for m in measure_ids) + f") AND data_value_type='{data_value_type}'"
    if state:
        where += f" AND stateabbr='{state}'"
    frames, offset = [], 0
    while True:
        chunk = pd.DataFrame(json.loads(_get(PLACES_COUNTY, {"$where": where, "$select": PLACES_SELECT,
                                                               "$limit": page, "$offset": offset})))
        frames.append(chunk)
        if len(chunk) < page:
            break
        offset += page
    df = pd.concat(frames, ignore_index=True)
    for c in ["data_value", "low_confidence_limit", "high_confidence_limit", "totalpopulation"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ---- Week 4 powerups (links checked 2026-10-26; 2022 is the latest SVI release) ----
SVI_COUNTY_URL = "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv"
def load_svi_county() -> pd.DataFrame:
    """CDC/ATSDR Social Vulnerability Index 2022, one row per county.
    Key columns: FIPS, STATE, COUNTY, E_TOTPOP, RPL_THEMES (overall percentile 0-1),
    RPL_THEME1..RPL_THEME4. The value -999 means 'not available'."""
    path = CACHE / "svi_2022_county.csv"
    if not path.exists():
        path.write_bytes(_get(SVI_COUNTY_URL, timeout=300))
    return pd.read_csv(path, dtype={"FIPS": str}).replace(-999, pd.NA)

CHR_URL = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
def load_county_health_rankings() -> pd.DataFrame:
    """County Health Rankings analytic file. Row 0 holds readable labels and row 1 the variable
    codes; we skip the labels so the codes (fipscode, v001..v1xx) become the column names.
    The data dictionary on the CHR site maps each v-code to a measure."""
    path = CACHE / "chr_analytic_2025_v3.csv"
    if not path.exists():
        path.write_bytes(_get(CHR_URL, timeout=300))
    return pd.read_csv(path, skiprows=[0], dtype={"fipscode": str, "statecode": str, "countycode": str})

print("course setup ready")


## 1. What your app must do
1. **One question, in plain words, at the top.** Your capstone question from week 2, sharpened by your week 6
   analysis plan. Write it the way your reader would ask it.
2. **The answer first.** A headline sentence with a number in it, in words a non-technical reader
   understands. It must change when the input changes.
3. **At least one input** (a `# @param` form field: dropdown, slider, text box) that lets the reader ask
   *their* version of the question: their state, their hospital, their condition. The input has to change the
   answer, not just filter a table nobody reads.
4. **At least one chart** that puts the answer in context: your reader's choice against the rest, or this
   period against the last.
5. **Honest measurement, on the page.** The denominator (*n* of what), what is missing or suppressed and
   why, a warning where numbers are small, and one sentence on what the measure does **not** mean.
6. **No code, no jargon.** Every code cell hidden (form view, section 2). Column names replaced with words.
   Any term an executive would ask about is defined in a line.
7. **Shared, and the link works**: a view-only link that opens for anyone at Fairfield.

Keep it to one screen, two at most. If the reader has to scroll to find the point, move the point up.

## 2. How to build a Colab app, step by step
A Colab notebook has one view, so the app is the notebook with its code hidden. You see code; your reader sees
the text, the form fields and the outputs. Two Colab features do the work: **forms** and **hidden code**.

**Inputs are form fields.** A comment `# @param` after an assignment turns it into a field above the code:
```python
state_choice = "CT"  # @param ["CT", "MA", "NY"] {run: "auto"}
min_hospitals = 10  # @param {type: "slider", min: 1, max: 50}
search = ""  # @param {type: "string"}
```
The first line makes a dropdown, the second a slider, the third a text box. Keep the `# @param` comment last
on its line; nothing may follow it.
- The options of a dropdown are typed in the comment, so they cannot come from a dataframe. Paste the list
  once (the example below lists every state in the file).
- `{run: "auto"}` re-runs the cell whenever the reader changes the field. It re-runs **only that cell**, not
  the cells below it. So put everything that depends on the choice inside one function, and call it from the
  form cell (the example's `show_app`). Otherwise the dropdown changes and the chart does not.

**Charts with plotly.** `plotly.express` makes an interactive chart in one line; a reader can hover for exact
values. Color the reader's choice and gray the rest, sort the bars, and write the title and axis labels in words
("Share of hospitals in the penalty range"), never column names. Use `yaxis_tickformat=".0%"` for shares.

**The headline.** `display(Markdown(f"## {headline}"))` prints a sentence as a large heading, and it changes
with the input because it is built inside `show_app`.

**Hide the code.** Put `# @title Short name` on the first line of a code cell. The cell becomes a form with that
title. Then open the cell's **⋮** menu (More cell actions) → **Form** → **Hide code**; double-clicking the form
toggles it too. Hide every code cell, setup and data prep included. Their outputs stay visible, so in your own
app make the prep cells print nothing (drop the `print` lines and the last-line dataframes).

**Publish = share a view-only link.** Readers see the outputs **saved** in the notebook, like Hex's published
snapshot. So: **Runtime → Run all**, check the page, **File → Save**. Then **Share** → under *General access*
choose **Fairfield University** (or *Anyone with the link*) with role **Viewer** → **Copy link**. Edits reach
readers only when you save again.

**What the reader can do.** Opening the link shows your page as saved; they can read it without running
anything. To change the dropdown they click **Runtime → Run all** (Colab may warn that the notebook was not
authored by Google: **Run anyway**). Their changes are never saved to your copy. Open your link in a private
browser window, or send it to your lab partner, and change the input once. If it works for them, it works.

**First load.** Run all reruns the downloads. If your data takes more than about 30 seconds to load, ask in
the course channel about caching it.

## 3. Worked example: how does my state compare?
**Question:** *Is a bigger or smaller share of my state's hospitals in Medicare's heart-failure readmission
penalty range than the nation's?* **Reader:** the director of a state hospital association, who decides where
to send readmission-reduction help.

The first two cells prepare the data; nothing in them depends on the reader's choice. The third defines
`show_app(state_choice)`, which builds the page for one state. The fourth is the reader's page: a dropdown and
a call to `show_app`. Run them in order.

In [ ]:
# @title Example data: one row per hospital in the HRRP heart-failure measure
hrrp = load_cms("hrrp")
hf = hrrp[hrrp["Measure Name"].eq("READM-30-HF-HRRP")].copy()
hf["excess_ratio"] = to_num(hf["Excess Readmission Ratio"])
hf["discharges"] = to_num(hf["Number of Discharges"])
# 1 = in the penalty range, 0 = not, missing = CMS published no ratio (missing is NOT "not penalized")
hf["penalized"] = (hf["excess_ratio"] > 1).astype(float).where(hf["excess_ratio"].notna())
hf["no_ratio_reason"] = hf["Footnote"].map({"1": "Too few cases to report",
                                            "5": "Not available for this period",
                                            "7": "No cases met the criteria"}).where(hf["excess_ratio"].isna())
hf_hospitals = hf.rename(columns={"Facility ID": "facility_id", "Facility Name": "facility_name",
                                  "State": "state"})[
    ["facility_id", "facility_name", "state", "excess_ratio", "discharges", "penalized", "no_ratio_reason"]]
PERIOD = f'{hf["Start Date"].iloc[0]} to {hf["End Date"].iloc[0]}'
print(len(hf_hospitals), "hospitals in the measure;", int(hf_hospitals["excess_ratio"].notna().sum()),
      "with a published ratio; period", PERIOD)


In [ ]:
# @title All states and the nation (they do not depend on the reader's choice)
by_state = sql("""
SELECT state,
       COUNT(*)                          AS hospitals_in_program,
       COUNT(excess_ratio)               AS hospitals_rated,
       CAST(SUM(penalized) AS INTEGER)   AS hospitals_penalized,
       ROUND(AVG(penalized), 3)          AS share_penalized,
       ROUND(MEDIAN(excess_ratio), 3)    AS median_ratio
FROM hf_hospitals
GROUP BY state
ORDER BY state
""")

national = sql("""
SELECT COUNT(*)                        AS hospitals_in_program,
       COUNT(excess_ratio)             AS hospitals_rated,
       CAST(SUM(penalized) AS INTEGER) AS hospitals_penalized,
       ROUND(AVG(penalized), 3)        AS share_penalized
FROM hf_hospitals
""")

print(len(by_state), "states and territories;", "national share in the penalty range:", f"{national['share_penalized'].iloc[0]:.1%}")


In [ ]:
# @title The page for one state: headline, chart, table
import plotly.express as px
from IPython.display import display, Markdown

HIGHLIGHT, OTHERS = "#2a6fdb", "#c5cbd3"

def show_app(state_choice: str) -> dict:
    """Everything that depends on the reader's choice, in one place, so {run: "auto"} redraws all of it."""
    my_state = sql(f"""
    SELECT * FROM by_state WHERE state = '{state_choice}'
    """)
    state_compare = sql(f"""
    SELECT state,
           share_penalized,
           hospitals_rated,
           CASE WHEN state = '{state_choice}' THEN 'Your state' ELSE 'Other states' END AS highlight
    FROM by_state
    WHERE hospitals_rated >= 10          -- states with fewer rated hospitals are too small to rank
    ORDER BY share_penalized DESC
    """)
    my_hospitals = sql(f"""
    SELECT facility_name                         AS hospital,
           ROUND(excess_ratio, 3)                AS excess_readmission_ratio,
           CAST(discharges AS INTEGER)           AS heart_failure_discharges,
           CASE WHEN excess_ratio IS NULL THEN no_ratio_reason
                WHEN excess_ratio > 1      THEN 'In penalty range'
                ELSE 'Not in penalty range' END  AS status
    FROM hf_hospitals
    WHERE state = '{state_choice}'
    ORDER BY excess_ratio DESC NULLS LAST
    """)

    me, us = my_state.iloc[0], national.iloc[0]
    gap = me["share_penalized"] - us["share_penalized"]
    if me["hospitals_rated"] < 10:
        compare = f"but with so few rated hospitals, a comparison with the nation ({us['share_penalized']:.0%}) is not reliable"
    elif abs(gap) < 0.05:
        compare = f"about the same as the nation ({us['share_penalized']:.0%})"
    else:
        compare = f"{'higher' if gap > 0 else 'lower'} than the nation ({us['share_penalized']:.0%})"
    headline = (f"In {state_choice}, {me['hospitals_penalized']} of {me['hospitals_rated']} rated hospitals "
                f"({me['share_penalized']:.0%}) are in Medicare's heart-failure readmission penalty range, {compare}.")
    rank_table = state_compare.reset_index(drop=True)
    rank = rank_table.index[rank_table["state"].eq(state_choice)]
    rank_note = (f"Rank {rank[0] + 1} of {len(rank_table)} states with at least 10 rated hospitals (1 = highest share)."
                 if len(rank) else "Not ranked: fewer than 10 rated hospitals.")
    unrated = int(me["hospitals_in_program"] - me["hospitals_rated"])

    display(Markdown(f"## {headline}\n{rank_note} {unrated} hospitals in the program have no published ratio."))

    fig = px.bar(state_compare, x="state", y="share_penalized", color="highlight",
                 color_discrete_map={"Your state": HIGHLIGHT, "Other states": OTHERS},
                 category_orders={"state": list(state_compare["state"])},
                 labels={"state": "State", "share_penalized": "Share of rated hospitals", "highlight": "",
                         "hospitals_rated": "Rated hospitals"},
                 hover_data={"hospitals_rated": True, "highlight": False},
                 title="Share of rated hospitals in the heart-failure penalty range, by state")
    fig.add_hline(y=us["share_penalized"], line_dash="dash", line_color="#555",
                  annotation_text=f"Nation {us['share_penalized']:.0%}", annotation_position="top right")
    fig.update_layout(yaxis_tickformat=".0%", showlegend=False, bargap=0.15, height=380,
                      plot_bgcolor="white", margin=dict(l=40, r=20, t=60, b=40))
    fig.update_yaxes(gridcolor="#e5e7eb")
    fig.show()

    display(my_hospitals.rename(columns={"hospital": "Hospital", "excess_readmission_ratio": "Excess readmission ratio",
                                         "heart_failure_discharges": "Heart failure discharges", "status": "Status"}))
    return {"headline": headline, "my_state": my_state, "state_compare": state_compare, "my_hospitals": my_hospitals}


## How does your state compare? *(the reader's page starts here)*
Pick a state. The headline, chart and table below change with it.

In [ ]:
# @title Pick a state
state_choice = "CT"  # @param ["AK", "AL", "AR", "AZ", "CA", "CO", "CT", "DC", "DE", "FL", "GA", "HI", "IA", "ID", "IL", "IN", "KS", "KY", "LA", "MA", "MD", "ME", "MI", "MN", "MO", "MS", "MT", "NC", "ND", "NE", "NH", "NJ", "NM", "NV", "NY", "OH", "OK", "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VA", "VT", "WA", "WI", "WV", "WY"] {run: "auto"}
app = show_app(state_choice)


**How to read this** *(app text, template)*: Each bar is a state. The height is the share of its rated
hospitals whose heart-failure readmissions were higher than Medicare expected for their patients. Your state is
highlighted; the dashed line is the nation.

**About the data** *(app text, template)*: CMS Hospital Readmissions Reduction Program, heart failure, *(period)*.
*n* hospitals in the program, *n* with a published ratio. *TODO: what is missing and why; what the measure does
not mean.*

## 4. Turn the example into an app (practice, about 15 minutes)
Do this once with the example before you build your own.
1. **Input.** Change the dropdown to `MA`. The headline, chart and table redraw, because the cell has
   `{run: "auto"}` and everything sits inside `show_app`. Then try `VT`. Does the page still make sense? (It
   should say the state is too small to rank.)
2. **Hide the code.** Every code cell above except the course setup cell already has a `# @title`. For each
   one, **⋮** → **Form** → **Hide code**. For the setup cell, **⋮** → **Form** → **Add form** first (it adds a
   title line), then **Hide code**. Scroll from the top: what is left is what your reader sees.
3. **Read it as the reader.** Is the question at the top? Is the answer the first thing you see after the
   dropdown? Fix the *How to read this* and *About the data* text above (the *TODO*).
4. **Share and test.** **Runtime → Run all**, **File → Save**, then **Share** → *General access* →
   **Fairfield University**, **Viewer** → **Copy link**. Open the link in a private window (sign in as your
   partner, or send it to them) and change the dropdown once after **Runtime → Run all**.

## 5. Your app
Build your midterm below, using the example as a pattern. Replace every *(edit)* and `# TODO`. Bring your
data-loading and cleaning code over from your capstone notebook (copy the cells, or rebuild them here).
Every variable name must be new: the example already uses `state_choice`, `show_app` and `app`.

# *(edit: your question, as your reader would ask it)*
*(edit: one line on who this is for and what decision it helps with)*

In [ ]:
# @title Load and prepare the data
# TODO: load and prepare your data (course setup helpers are available). End with one tidy dataframe.
# Print nothing here in your final version: the reader sees every output.


In [ ]:
# @title The page for one choice
# TODO: my_app(choice) runs the query (sql() or pandas) for that choice, builds my_headline (one sentence, a
# number, a comparison, in plain words), shows it with display(Markdown(f"## {my_headline}")), draws a plotly
# chart that highlights the choice, and shows at most one table with column names in words.
from IPython.display import display, Markdown

def my_app(my_choice):
    my_headline = "TODO"
    display(Markdown(f"## {my_headline}"))


In [ ]:
# @title Your input
# TODO: rename the options (and the title above) to suit your question. Keep {run: "auto"}.
my_choice = "TODO"  # @param ["TODO"] {run: "auto"}
my_app(my_choice)


**How to read this:** *(edit: one or two sentences on the chart)*

**About the data:** *(edit: source and period; n of what; what is missing, suppressed or excluded, and how many;
what the measure does not mean)*

## 6. Rubric
Each criterion is scored 4 to 0; the midterm is out of 20 and is worth 15% of the course grade.

| Criterion | 4 | 3 | 2 | 1 | 0 |
|---|---|---|---|---|---|
| **Answers the question** | The question is stated in plain words at the top; the headline answers it with a specific number and a comparison; the analysis behind it is sound. | Question and answer are there, but the reader has to work a little (the answer is low on the page, or a number has no comparison). | The app shows relevant data, but the reader has to work out the answer. | The question is vague, or the app answers a different one. | No question, or nothing answers it. |
| **Inputs** | At least one input changes the answer: headline, chart and table all respond. Sensible default, options in words, and no choice leaves a broken or empty page. | The input works and changes the answer, with small problems (codes instead of names, one choice gives an empty chart). | The input only filters a table; the headline and chart do not respond. | An input is there but broken or decorative. | No input. |
| **Honest measurement** | Denominators stated; missing, suppressed and excluded data named with counts and reasons; small-number warning where it applies; one sentence on what the measure does not mean. | Most of that is there, with one gap. | The source is named, but denominators or missing data are not addressed. | Numbers without context, or a misleading comparison (tiny groups ranked with no warning). | Wrong numbers, or an error that changes the answer. |
| **Clarity for a non-technical reader** | Answer first; plain words with no code or column names; the chart is titled and labeled in words and draws the eye to the reader's choice; fits on one screen; a "how to read this" line. | Clear, with some jargon or clutter. | The reader would need you there to explain it; code or raw column names are visible. | Mostly a notebook dump. | Not shared, or the view-only link does not open. |
| **Ownership** | You can explain every cell behind the app and every number on the page, including how it could be wrong. | You can explain nearly everything. | You can explain the app but not some of the code behind it. | You cannot explain key numbers or code. | The work is not your own. |

## 7. Submission checklist
- [ ] The question is at the top of the app, in plain words.
- [ ] The headline is a sentence with a number and a comparison, and it changes with the input.
- [ ] At least one input (a `# @param` field with `{run: "auto"}`); every option gives a page that makes sense
      (try the smallest one).
- [ ] At least one chart, titled and labeled in words, highlighting the reader's choice.
- [ ] *About the data*: source, period, *n*, what is missing or excluded and why, what the measure does not mean.
- [ ] Every code cell hidden (form view); no raw column names visible; prep cells print nothing.
- [ ] **Runtime → Run all**, then **File → Save**, so the saved outputs are the latest (save again after changes).
- [ ] The view-only link opens for someone else (your lab partner checked it).
- [ ] Notebook title `Lab 8 — <your last name>`; a pinned revision saved (**File → Save and pin revision**).
- [ ] Notebook shared with the instructor (**Editor**), and the view-only app link posted in the course
      submission form.
- [ ] You can explain every cell (Ownership).

## 8. Powerup: deploy the same analysis as a Streamlit app
Streamlit turns a Python script into a web app. It runs on your own computer, which is the point: you will
set up a local Python environment with **uv** and ship the analysis outside the notebook.

1. Install uv (one line from https://docs.astral.sh/uv/getting-started/installation/), then in a terminal:
   ```
   uv init midterm-app
   cd midterm-app
   uv add streamlit pandas duckdb plotly
   ```
2. Save the course setup cell (the first code cell of any lab) as `course_data.py` in that folder.
   (**File → Download → Download .py** gives you this whole notebook as a script to copy from.)
3. Write `app.py`: load your data with the same code as your Colab notebook, add `st.selectbox(...)` for the
   input, `st.subheader(headline)` for the answer, `st.plotly_chart(fig)` (or `st.bar_chart(...)`) for the
   chart, `st.dataframe(...)` for the table, and `st.markdown(...)` for the notes. Wrap the data loading in a
   function decorated with `@st.cache_data` so it runs once, not on every click.
4. Run it: `uv run streamlit run app.py`. A browser tab opens at `http://localhost:8501`.
5. Submit a screenshot and your `app.py` (paste it into a text cell in this notebook). Deploying it to the
   web (Streamlit Community Cloud) is optional and needs a public GitHub repository: never put keys or private
   data in it.